# Dreamer V1 persistence probe
This notebook only tests whether a small checkpoint and CSV survive Kaggle Save & Run All output persistence. It does not train Dreamer.


In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch
run_id = 'persistence_probe_20261024_01'
root = Path('/kaggle/working') / ('dreamer_' + run_id)
root.mkdir(parents=True, exist_ok=True)
model = torch.nn.Linear(4, 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
x = torch.ones(3, 4)
loss = model(x).square().mean()
loss.backward()
opt.step()
ckpt = root / 'probe.pt'
torch.save({'run_id': run_id, 'model': model.state_dict(), 'optimizer': opt.state_dict()}, ckpt)
(root / 'probe.csv').write_text('run_id,loss\n' + f'{run_id},{loss.item()}\n', encoding='utf-8')
child = root / 'load_probe.py'
child.write_text('import torch\n' + f'p=torch.load(r"{ckpt}", map_location="cpu")\n' + 'print("fresh_process_load=PASS run_id=" + p["run_id"])\n', encoding='utf-8')
result = subprocess.run([sys.executable, str(child)], capture_output=True, text=True)
(root / 'load_probe.log').write_text(result.stdout + result.stderr, encoding='utf-8')
if result.returncode != 0: raise RuntimeError(result.stderr)
bundle = root / 'persistence_probe_bundle.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z: z.write(ckpt, ckpt.name); z.write(root/'probe.csv', 'probe.csv'); z.write(root/'load_probe.log', 'load_probe.log')
print(json.dumps({'run_id': run_id, 'bundle': str(bundle), 'fresh_process': result.stdout.strip(), 'bytes': bundle.stat().st_size}, indent=2))
